#環境構築

データの暗号化・複合化を可能にするライブラリ、pycryptodomeをインストール。

In [1]:
pip install pycryptodome

#RSA鍵を生成する

In [2]:
from Crypto.PublicKey import RSA

# 秘密鍵の生成
private_key = RSA.generate(2048)
with open('private.pem', 'w') as f:
    f.write(private_key.export_key().decode('utf-8'))
 
# 秘密鍵に基づいて公開鍵を生成
public_key = private_key.publickey()
with open('public.pem', 'w') as f:
    f.write(public_key.export_key().decode('utf-8'))

# pemファイルをロード
with open('private.pem', 'br') as f:
    private_pem = f.read()
    private_key = RSA.import_key(private_pem)
with open('public.pem', 'br') as f:
    public_pem = f.read()
    public_key = RSA.import_key(public_pem)

# nとeを16進数に変換
n = hex(public_key.n)
e = hex(public_key.e)

# 出力
print(n)
print(e)

0xdb5ba8b19590687257c509055a8a1c2b75228ed2b072d4c0ca9ece3a9a3b2b0e119df108fb64e7c7a69721b13676dbc57b311ea8a61ed6c3b5312a38d11f8099f91fc93f98bdd9c3714b0661a391bead25f36b6fc3692716b9b7f77587a25c53b69b4aa3e9dcf95d47e1b406680a47de774c7df340cee171902520d2acacaec3352303414d0c75b2976591ec10d520312c52f842bce7a7b87e2e21d5db4d249b05a98312996949a92abdf05cd014a12541dc9361d5a626a2942d4d487ca15ea4c443db7f8f81b6752afdf9e7c36d527015e3a6d5ebf2eaf269883c2bf92512db3916251a3975c92a45c18563657edc6fb53c554263574db8ef8ab123411d4c21
0x10001


#RSA署名を作成する

In [3]:
from Crypto.Hash import SHA256
from Crypto.Signature import pkcs1_15

# メッセージと秘密鍵から署名を生成
message = '情報セキュリティ'
h1 = SHA256.new(message.encode())
signature = pkcs1_15.new(private_key).sign(h1)

# 16進数に変換
signature16 = hex(int.from_bytes(signature, byteorder='big'))

# 出力
print(signature16)

0x29c874b01f48d7e81f11d49838108bb0ecb80fdf842669152262141ec9639a1aa5f2fea3cf8f9b6405c6be3e06caa157bf1a630b6c349f3f4c9ef62ffd0dd0b6e8342a1400ba7b43960c0956d27bfdeebebe7a21984e9569271a5fbc9576556d95926d6009f086165172797de6b5a10cd180252aa163e6ab2bb0986adc60fedbff71408fbc101f701ed6eabb551603610e1438ab20ca3f5c607118b950b7807984606e221e60c53454473a9541bb908b3193a2049b479b87ade3dd80e08b87a40934751872f33f40fc37f1bfbf33622c4be8a3c5802cf5e95d8632070ba0bd8a9e47f60f100548608171e1d97352143112dde1be7666fd2c2272fa333f9489b7


#RSA署名を検証する

In [4]:
import binascii

# 検証する関数
def verification(n, e, signature):
  # 公開鍵(n,e)を16進数 → int型に変換
  int_n = int(n, 16)
  int_e = int(e, 16)
  public_key = RSA.construct((int_n, int_e))
  # 署名を16進数 → バイトコードに変換
  int_sig = int(signature, 16)
  format_sig = format(int_sig, 'x')
  binascii_sig =  binascii.unhexlify(format_sig)
  # 公開鍵から妥当性を検証
  message = '情報セキュリティ'
  h2 = SHA256.new(message.encode())
  try:
    pkcs1_15.new(public_key).verify(h2, binascii_sig)
    verified = True
  except ValueError:
    verified = False
  print(verified)


In [5]:
# 他の学生の署名を検証(195735A)
n1 = "0xd43c10a381ab6e9cf33dd0575f05b01005b0e8d4aa610fc1a57721497e6948c8db51830b8fc4290f3f2008a1ac96d27a699882c353f946c77cc94bb293cfd38bc5c41111a32aed0fdafa1529718a74c2929990e0e191556e61e7121b82d430c4559629d17f1c10ecfa826c44bc732f34c8992a55c471739e6d8761c3fb023dcf7f756bf62ed360e89d6bc392ba75e5ad4dab80b63509dc848d8a2515111171c0cc3d39249c5c2a1b42d27dc7f1666f0d571102672da41352e40c5ea38a82159ce0b75cfbf75daa54b8afb7ea47cdc67bb978d1b5486b905af54730172fcb58a1b8e761283f0c2143d6531827e8854297c28394bbd0b0b43cd57a8179f88bc301"
e1 = "0x10001"
sig1 = "0x85ce114f8837cdc4bac991d454caef7e9b17eecf2098647b737e666e48bdc55ec08a98f718984e9aa1f5a6e692155bc806e520f1dc9564d27c45f5a9c94cc09933d6ea40096767216d7392c4c9237a3060e35b0233ed55fe008d84edd39111dc068b9953f154a2a52031d561ae5f9aba7f0ac065205b4ec8494fda0e6f90fe68b22f3d99a52d5cb800e686b34386db6ac87359b34b5f03d5f06760e98d2b04352dcb28c860f2a44322e68d055f762e230f55f27eb8f4d8d63f3a8108a5b549a835de666e2158adc573cd022e2b354d5bf2d0cf973a0d3313d77552e178d1a9edc95a86a5285960de1af6119865457ec374d79b78c20c6f3a056a0c7db3aa9d81"

verification(n1, e1, sig1)

True


In [6]:
# 他の学生の署名を検証(195265B)
n2 = "0xd0aaa164a4fab534b78530d264d0c28e1097db5f912b0696c5675758844abc62b0825353a8c3dcaabbc50b0015ea60f90162dd5503da4436201a95b1569e01890ae83bafbc35264cfe413fde6c8ead10d35b44793d63b85d8cb2b6c24e260f4a4fc4bd53110cdd23b096d873ba2fbb358871b32c2027743b67bf773db18509a5eb7cc4c1d5cb582c2f6bff648f379fbac8936f339054f9806e854366b8186136956b1c34735bef6f4cc2db3798931e0915f7d8f8f125afcda33b6f9c73d817c81c5596bcc4e7bf437036cdc928891b16ca2c7b47f23e0d2857c733a65d2e1bf39948b64a92206421d8143bfc627bcc7d14f4079099232cf166cc54d6de046d03"
e2 = "0x10001"
sig2 = "0x96c067ee5000b6306cdf8fb6d3b8c81d145272bfeec0dab0828f053ae33b93aa0540a9e3af82c99b9c4c3ec55a330bb7ea61a9aca6df333b53cf250e420c77f65e7db82ded0fabce51c54e527956d65563fb9ca66ade6f84436957941cb0007303c8b2133b42a0309a0a239e5ac94a9e92ab95626567687a99c989ed1d474fdb97a0769d8183b7f4cd480cf386561382a2aebc9b3c154c09c32d18f096117325b8a83785dd7078cc2f8bf5ef2b1478a051014aa00c5226b7f8cd26c19fbab1da8664216b9ec6bad9e25e844614f0f57c0706f6416a545976eedb1beb43a4b3d4947d58ef3054aa9ad633c2fc0fb847068c3c167bdb654ea6d1aed03d6482b98a"

verification(n2, e2, sig2)

True


In [7]:
# 他の学生の署名を検証(195244H)
n3 = "0xb5248014587fa4b53f137dddc1c461ca7b1e59bc056f21a04c09974284cd7fae210c82e8fbfe5c05c4e1e77ebef5b3f8f51fed9317d2619db6c661d6ec0a7d41fb7a922f68c2c32f844340582ea44e45c9557b35fa08508bd5226d4ba947a87ad48a73b9a255c71f3a4a6739086a99be4579df3f4700c2b55ae4c1b6299d07c7f84da3a50888cf8ea8a91b8d8d4a40abd3cdeaba875e167faec691d783488d03bd0d290e37d9b067507f0bc63808253390ed69f6cac2a0c66b4f4a506b4999560ae201f1c7f3d6e8b7c5b9b38332116ae7e0682478d84875c86f45a736de43f50ee5f0541dd957b413b1d668a1ff1b68a7afcfd0efcd0c2f57885095356026c5"
e3 = "0x10001"
sig3 = "0xbed20deff8ae0cc5847ae5f00630934ef36234280eb22c6187a7bf5698facf0c830c4df66068471a5fc8e219a7239c09dcc62740e21bd6a990bad96f89f44b37a8c2e99225835b382eeb076aaf0faf6828ab566af2e67db61c237f1cbc4be1b27ca3cbc22b64fc596d768b41dbcbd7ac0714e4507f61a69ba051be6ab33387816f0ed9154161bd2b65a3332cd9321900e51afd4e3f74fbef205fc1179db6bd7d3d00855e9fe647688362424273c83b520c14454f01e3f9d9d04a3efaaa87f53c71a704bb329c2ccdbdfd3486569d266d7e013c4cfd08ce9a7d75448616d633c382649a58c31513f525906d56967708281165725c55205f3af61ddf4e25afbf"

verification(n3, e3, sig3)

False
